# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 22 05:51:19 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,766041,41.0,1473300,78.7,1473300,78.7
Vcells,1510570,11.6,127755098,974.7,155529138,1186.6


In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [ ]:
# semilla primigenia
PARAM <- list()
PARAM$semilla_primigenia <- 100103

# cp fijo
PARAM$cp <- -1

# cantidad maxima de arboles por configuracion
PARAM$num_trees_max <- 32

# grid de hiperparametros
PARAM_GRID <- list(
  feature_fraction = c(0.50, 0.75),
  minsplit = c(200, 400, 600),
  divisor_minbucket = c(2, 3),
  maxdepth = c(4, 6, 8)
)

# genero todas las combinaciones (minsplit y su divisor quedan pegados fila a fila)
grid <- expand.grid(PARAM_GRID)

# minbucket se calcula DESPUES, fila por fila
grid$minbucket <- floor(grid$minsplit / grid$divisor_minbucket)
grid$divisor_minbucket <- NULL

grid <- unique(grid)

# elimino combinaciones donde minbucket >= minsplit (resguardo)
grid <- grid[grid$minbucket < grid$minsplit, ]

message("Cantidad de combinaciones: ", nrow(grid))

Cantidad de combinaciones: 36



In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4020_OK"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria", "numero_de_cliente")))

In [ ]:
# que tamanos de ensemble grabo a disco
grabar <- c(1, 2, 4, 8, 16, 32)

### Reanudacion automatica

Ahora se recorren TODAS las combinaciones de `grid`, no una unica configuracion fija. Se guarda un checkpoint despues de cada arbol y un registro en el log cuando una configuracion completa los `PARAM$num_trees_max` arboles. Si la sesion se corta, al volver a correr la celda de abajo se retoma automaticamente desde el ultimo checkpoint (no se pierde el trabajo ya hecho).

El resto del comportamiento es igual al notebook original: se entrena siempre sobre el 100% de `dtrain` y se predice sobre `dfuture`. La diferencia es que en cada punto de `grabar` **solo se genera el CSV** (ya no se sube a Kaggle automaticamente); la subida se hace al final, eligiendo manualmente que configuracion enviar (ver la seccion "Subida manual a Kaggle" mas abajo).

In [ ]:
#--------------------------------------------------
# Reanudacion automatica: log y checkpoints
#--------------------------------------------------

log_file <- "/content/buckets/b1/exp/z420_hyperparametros_log.csv"

checkpoint_dir <- "/content/buckets/b1/exp/z420_checkpoints"
dir.create(checkpoint_dir, recursive = TRUE, showWarnings = FALSE)

if (file.exists(log_file)) {
  log_experimentos <- fread(log_file)
  message("Se encontro log previo con ", nrow(log_experimentos), " registros.")
} else {
  log_experimentos <- data.table(
    feature_fraction = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    num_trees = integer(),
    status = character(),
    checkpoint = character()
  )
  message("No existe log previo. Se comienza desde cero.")
}

experimento_completado <- function(ff, ms, mb, md) {
  if (nrow(log_experimentos) == 0) return(FALSE)

  any(
    log_experimentos$feature_fraction == ff &
    log_experimentos$minsplit == ms &
    log_experimentos$minbucket == mb &
    log_experimentos$maxdepth == md &
    log_experimentos$status == "completed"
  )
}

# base de la tabla de prediccion (se reutiliza para cada configuracion)
tb_prediccion_base <- dfuture[, list(numero_de_cliente)]

No existe log previo. Se comienza desde cero.



In [ ]:
#--------------------------------------------------
# Loop principal: recorro todas las combinaciones del grid
#--------------------------------------------------

for (i in seq_len(nrow(grid))) {

  feature_fraction_val <- grid$feature_fraction[i]
  minsplit_val <- grid$minsplit[i]
  minbucket_val <- grid$minbucket[i]
  maxdepth_val <- grid$maxdepth[i]

  experimento_actual <- paste0(
    "ff", format(feature_fraction_val, nsmall = 2),
    "_ms", minsplit_val,
    "_mb", minbucket_val,
    "_md", maxdepth_val
  )

  checkpoint_file <- file.path(
    checkpoint_dir,
    paste0(experimento_actual, ".rds")
  )

  # Si ya termino esta configuracion, la salto.
  if (experimento_completado(
    feature_fraction_val,
    minsplit_val,
    minbucket_val,
    maxdepth_val
  )) {
    message("SKIP configuracion ya completada: ", experimento_actual)
    next
  }

  message(
    "\n=============================================\n",
    "Configuracion ", i, "/", nrow(grid), "\n",
    experimento_actual,
    "\n============================================="
  )

  #------------------------------------------------
  # Recupero checkpoint si existe
  #------------------------------------------------

  if (file.exists(checkpoint_file)) {

    checkpoint <- readRDS(checkpoint_file)

    arbol_inicio <- checkpoint$arbol_siguiente
    prob_acumulada <- checkpoint$prob_acumulada

    if (!is.null(checkpoint$rng_state)) {
      .Random.seed <- checkpoint$rng_state
    }

    message("REANUDANDO desde arbol ", arbol_inicio)

  } else {

    arbol_inicio <- 1L
    prob_acumulada <- rep(0, nrow(dfuture))

    # Una semilla distinta y reproducible por configuracion.
    set.seed(PARAM$semilla_primigenia + i)

    message("NUEVA configuracion. Seed = ", PARAM$semilla_primigenia + i)
  }

  PARAM$rpart <- list(
    cp = PARAM$cp,
    minsplit = minsplit_val,
    minbucket = minbucket_val,
    maxdepth = maxdepth_val
  )

  #------------------------------------------------
  # Genero los arboles
  # (igual que el notebook original: entreno sobre el
  #  100% de dtrain, predigo sobre dfuture)
  #------------------------------------------------

  if (arbol_inicio <= PARAM$num_trees_max) {

    for (arbolito in arbol_inicio:PARAM$num_trees_max) {

      message(
        "  ", experimento_actual,
        " - arbol ", arbolito,
        "/", PARAM$num_trees_max
      )

      qty_campos_a_utilizar <- as.integer(
        length(campos_buenos) * feature_fraction_val
      )

      campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
      campos_random <- paste(campos_random, collapse = " + ")

      formulita <- paste0("clase_ternaria ~ ", campos_random)

      modelo <- rpart(
        formulita,
        data = dtrain,
        xval = 0,
        control = PARAM$rpart
      )

      prediccion <- predict(modelo, dfuture, type = "prob")

      prob_acumulada <- prob_acumulada + prediccion[, "BAJA+2"]

      if (arbolito %in% grabar) {

        umbral_corte <- (1 / 40) * arbolito

        tb_prediccion <- copy(tb_prediccion_base)
        tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

        archivo_kaggle <- paste0(
          "KA420_", experimento_actual,
          "_nt", sprintf("%.3d", arbolito),
          ".csv"
        )

        fwrite(
          tb_prediccion[, list(numero_de_cliente, Predicted)],
          file = archivo_kaggle,
          sep = ","
        )

        message("    Archivo Kaggle generado: ", archivo_kaggle)

        # NOTA: ya no se sube a Kaggle aca. Los CSV quedan
        # generados en disco y la subida se hace al final,
        # eligiendo manualmente cual configuracion enviar.
      }

      #------------------------------------------------
      # CHECKPOINT DESPUES DE CADA ARBOL
      #------------------------------------------------
      saveRDS(
        list(
          arbol_siguiente = arbolito + 1L,
          prob_acumulada = prob_acumulada,
          rng_state = .Random.seed
        ),
        file = checkpoint_file
      )
    }
  }

  #------------------------------------------------
  # Configuracion completa: registro en el log
  #------------------------------------------------

  log_entry <- data.table(
    feature_fraction = feature_fraction_val,
    minsplit = minsplit_val,
    minbucket = minbucket_val,
    maxdepth = maxdepth_val,
    num_trees = PARAM$num_trees_max,
    status = "completed",
    checkpoint = checkpoint_file
  )

  fwrite(
    log_entry,
    file = log_file,
    append = TRUE,
    col.names = !file.exists(log_file)
  )

  # Conservo el checkpoint como respaldo historico.
  file.rename(
    checkpoint_file,
    paste0(checkpoint_file, ".completed")
  )

  message("COMPLETADA: ", experimento_actual)
}


Configuracion 1/36
ff0.50_ms200_mb100_md4

REANUDANDO desde arbol 6

  ff0.50_ms200_mb100_md4 - arbol 6/32



## Verificacion de ganancia

In [ ]:
#--------------------------------------------------
# GANANCIA de un archivo de corrida ya generado
#--------------------------------------------------
# Compara el CSV (numero_de_cliente, Predicted) contra el
# clase_ternaria real de esos clientes (recuperado del
# 'dataset' original, que nunca se pisa con NA).

calcular_ganancia_archivo <- function(archivo_csv, dataset_completo, mes_test = 202109) {

  predicciones <- fread(archivo_csv)

  reales <- dataset_completo[
    foto_mes == mes_test,
    list(numero_de_cliente, clase_ternaria)
  ]

  comparacion <- merge(predicciones, reales, by = "numero_de_cliente")

  ganancia <- comparacion[
    ,
    sum(
      ifelse(
        Predicted == 1,
        ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
        0
      )
    )
  ]

  ganancia
}

#--------------------------------------------------
# Uso: indicar el nombre del archivo a revisar
#--------------------------------------------------

#archivo_a_revisar <- "KA420_ff0.25_ms200_mb100_md4_nt032.csv"

#ganancia_local <- calcular_ganancia_archivo(archivo_a_revisar, dataset)

#cat("\nGanancia estimada para", archivo_a_revisar, ":", ganancia_local, "\n")

In [ ]:
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
setwd("/content/buckets/b1/exp/exp4020_OK")

In [ ]:
#--------------------------------------------------
# Uso: indicar el nombre del archivo a revisar
#--------------------------------------------------

archivo_a_revisar <- "KA420_ff0.50_ms200_mb100_md4_nt001.csv"

ganancia_local <- calcular_ganancia_archivo(archivo_a_revisar, dataset)

cat("\nGanancia estimada para", archivo_a_revisar, ":", ganancia_local, "\n")


Ganancia estimada para KA420_ff0.50_ms200_mb100_md4_nt001.csv : -257975000 


## Subida manual a Kaggle

Como esta version entrena siempre sobre el 100% de `dtrain` (no hay `dvalid` ni ganancia local calculada), no hay forma automatica de saber cual configuracion es "la mejor". Los CSV de todas las combinaciones y todos los puntos de `grabar` ya estan generados en la carpeta del experimento (`/content/buckets/b1/exp/exp4020/`).

Elegi manualmente los hiperparametros y la cantidad de arboles que queres subir, completando la proxima celda, y despues corre la celda de envio.

In [ ]:
#--------------------------------------------------
# CONFIGURACION ELEGIDA MANUALMENTE PARA KAGGLE
#--------------------------------------------------
# Completar con los valores de la combinacion que se
# quiere subir (tienen que ser valores que existan en
# 'grid' y en 'grabar').

feature_fraction_elegida <- 0.25
minsplit_elegido <- 200
minbucket_elegido <- 100
maxdepth_elegido <- 4
num_trees_elegido <- 32

experimento_elegido <- paste0(
  "ff", format(feature_fraction_elegida, nsmall = 2),
  "_ms", minsplit_elegido,
  "_mb", minbucket_elegido,
  "_md", maxdepth_elegido
)

archivo_kaggle <- paste0(
  "KA420_", experimento_elegido,
  "_nt", sprintf("%.3d", num_trees_elegido),
  ".csv"
)

cat("\nArchivo elegido:\n", archivo_kaggle, "\n")

if (!file.exists(archivo_kaggle)) {
  stop(
    "No se encontro el archivo esperado: ", archivo_kaggle,
    "\nRevisar que la busqueda para esta configuracion haya",
    " llegado a ese punto de 'grabar'."
  )
}

In [ ]:
#--------------------------------------------------
# ENVIO A KAGGLE
#--------------------------------------------------

comando <- "kaggle competitions submit"
competencia <- "-c utn-2026-inicial"
arch <- paste("-f", archivo_kaggle)
mensaje <- paste0(
  "-m '", experimento_elegido,
  " cp=", PARAM$cp,
  " minsplit=", minsplit_elegido,
  " minbucket=", minbucket_elegido,
  " maxdepth=", maxdepth_elegido,
  " num_trees=", num_trees_elegido,
  "'"
)

linea <- paste(comando, competencia, arch, mensaje)

cat("\nComando a ejecutar:\n", linea, "\n")

# Descomentar para subir de verdad:
# salida <- system(linea, intern = TRUE)
# cat(salida)

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")



---

